# Least-squares (χ²) fit with iminuit — hands-on

**Goal:** fit a straight line $y=\theta_0+\theta_1 x$ to 10 data points with known uncertainties, and learn to read the result: best-fit values, covariance, goodness of fit, confidence regions and the propagated uncertainty band.

[iminuit](https://iminuit.readthedocs.io) is a Python interface to the Minuit2 library (CERN). Minuit minimises a function of several parameters and gives the parameter errors from the curvature of that function at the minimum.

**How to use this notebook.** Run the cells in order. Cells marked **✏️ Exercise** contain `...` that you must fill in. Cells marked **❓ Question** ask you to interpret what you see — write your answer in the cell below it.

## 0. Setup

In [ ]:
# Uncomment on Colab / a fresh environment:
# !pip install iminuit

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy import stats
import iminuit
from iminuit import Minuit

print("iminuit version:", iminuit.__version__)
plt.rcParams.update({"font.size": 13, "figure.figsize": (6, 4.5)})

## 1. The data

Ten measurements $(x_i, y_i)$, each $y_i$ with a known Gaussian uncertainty $\sigma_i$.

In [ ]:
xd     = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0])
yd     = np.array([1.1, 1.2, 1.5, 1.8, 2.1, 2.3, 2.6, 3.1, 3.5, 4.0])
yd_err = np.array([0.5, 0.6, 0.2, 0.4, 0.3, 0.2, 0.4, 0.3, 0.4, 0.5])

plt.errorbar(xd, yd, yerr=yd_err, fmt="bo")
plt.xlabel("x"); plt.ylabel("y"); plt.xlim(0, 6); plt.ylim(0, 6)
plt.show()

**❓ Question 1.** By eye, what slope and intercept would you expect? Write your guess here: *(your answer)*

## 2. Model and χ² function

Model: $f(x;\theta_0,\theta_1)=\theta_0+\theta_1 x$. The least-squares cost function is

$$\chi^2(\theta_0,\theta_1)=\sum_{i=1}^{N}\left(\frac{y_i-f(x_i;\theta_0,\theta_1)}{\sigma_i}\right)^2 .$$

In [ ]:
def f(x, theta0, theta1):
    ...   # straight line

def chi2_fcn(theta0, theta1):
    ...   # sum of squared normalised residuals

print(f"chi2(1.0, 0.5) = {chi2_fcn(1.0, 0.5):.2f}")

✏️ **Exercise 1.** Implement `f` and `chi2_fcn`. *(Check: χ²(1.0, 0.5) should be about 5.04.)*

### 2b. Feel the minimum: scan χ² by hand

Fix $\theta_0$ at 0.5 and scan $\theta_1$. Near the minimum χ² is a parabola, and the parameter uncertainty is the distance from the minimum where χ² has risen by 1: $\chi^2(\hat\theta\pm\sigma)=\chi^2_{\min}+1$.

In [ ]:
th1_scan = np.linspace(0.4, 0.9, 200)
chi2_scan = ...   # chi2 for each value of theta1, with theta0 fixed at 0.5

plt.plot(th1_scan, chi2_scan)
plt.axhline(chi2_scan.min() + 1, color="gray", ls="--", label=r"$\chi^2_{min}+1$")
plt.xlabel(r"$\theta_1$"); plt.ylabel(r"$\chi^2(\theta_0=0.5,\theta_1)$"); plt.legend()
plt.show()

✏️ **Exercise 2.** Fill in `chi2_scan`. 

**❓ Question 2.** Read off the two $\theta_1$ values where the curve crosses the dashed line. Their half-distance is the error on $\theta_1$ **with $\theta_0$ held fixed**. Will the error from the full fit (both parameters free) be larger or smaller? Why? *(your answer)*

## 3. Minimise with iminuit

Minuit takes the function and reads the parameter names from its signature. `errordef` tells Minuit what Δ(function) corresponds to "1σ": for a χ² it is 1 (`Minuit.LEAST_SQUARES`); for a *negative log-likelihood* it would be 0.5 (`Minuit.LIKELIHOOD`).

In [ ]:
m = Minuit(chi2_fcn, theta0=1.0, theta1=0.5)   # start values
m.errordef = ...
...              # run the minimiser (hint: a method of Minuit)
m

✏️ **Exercise 3.** Set `errordef` and run the minimiser. Check that `m.valid` is `True` in the output table.

In [ ]:
for p in m.parameters:
    print(f"{p} = {m.values[p]:.4f} +/- {m.errors[p]:.4f}")
print(f"chi2_min = {m.fval:.3f}")

### 3b. Cross-check: the linear problem has an exact solution

For a model linear in the parameters, $y = A\theta$, the weighted least-squares solution is

$$\hat\theta = C\,A^{T}W y,\qquad C=(A^{T}WA)^{-1},\qquad W=\mathrm{diag}(1/\sigma_i^2),$$

and $C$ *is* the covariance matrix of $\hat\theta$.

In [ ]:
A = np.vstack([np.ones_like(xd), xd]).T     # design matrix, shape (10, 2)
W = ...                                      # diagonal weight matrix
C_an  = ...                                  # covariance matrix
th_an = ...                                  # best-fit parameters

print("analytic theta :", th_an)
print("analytic errors:", np.sqrt(np.diag(C_an)))
print("agrees with Minuit?", np.allclose(th_an, m.values) and np.allclose(C_an, m.covariance, rtol=1e-3))

✏️ **Exercise 4.** Implement the normal equations. The last line must print `True`.

## 4. Plot the fit

In [ ]:
theta0, theta1 = m.values["theta0"], m.values["theta1"]
xf = np.linspace(0, 6, 300)
yf = f(xf, theta0, theta1)

n_dof = len(xd) - m.nfit
fit_info = [rf"$\chi^2$ / $n_\mathrm{{dof}}$ = {m.fval:.1f} / {n_dof}"]
for p, v, e in zip(m.parameters, m.values, m.errors):
    fit_info.append(rf"{p} = ${v:.3f} \pm {e:.3f}$")

fig, ax = plt.subplots()
ax.errorbar(xd, yd, yerr=yd_err, fmt="bo", label="data")
ax.plot(xf, yf, "r--", label="fit")
ax.set(xlabel="x", ylabel="y", xlim=(0, 6), ylim=(0, 6))
ax.legend(title="\n".join(fit_info), loc="upper left")
plt.tight_layout(); plt.show()

## 5. Goodness of fit

If the model is right and the errors are right, $\chi^2_{\min}$ follows a $\chi^2$ distribution with $n_\mathrm{dof}=N-N_\mathrm{par}$ degrees of freedom (expectation $=n_\mathrm{dof}$). The **p-value** is the probability of getting a χ² *at least as large* as observed:

$$p=\int_{\chi^2_{\min}}^{\infty}f_{\chi^2}(t;n_\mathrm{dof})\,dt .$$

In [ ]:
chi2_min = m.fval
pvalue = ...   # hint: scipy.stats.chi2 has .sf(x, df)
print(f"chi2/ndf = {chi2_min:.2f} / {n_dof} = {chi2_min/n_dof:.2f},  p-value = {pvalue:.3f}")

✏️ **Exercise 5.** Compute the p-value.

**❓ Question 3.** χ²/ndf ≈ 0.16 and p ≈ 0.996. Is this a *good* fit? A p-value this close to 1 means the points scatter around the line **much less** than their error bars say they should. Give two possible explanations. *(your answer)*

In [ ]:
x = np.linspace(0, 20, 1000)
pdf = stats.chi2.pdf(x, df=n_dof)

fig, ax = plt.subplots()
ax.plot(x, pdf, "b-")
x_hi = x[x >= chi2_min]
ax.fill_between(x_hi, stats.chi2.pdf(x_hi, df=n_dof), color="b", alpha=0.2, label=f"p = {pvalue:.3f}")
ax.axvline(chi2_min, color="k", ls="--", label=r"observed $\chi^2_{min}$")
ax.set(xlabel=r"$\chi^2$", ylabel=rf"$f(\chi^2;\,n_{{dof}}={n_dof})$", xlim=(0, 20))
ax.legend(); plt.show()

### 5b. Check the p-value with toy experiments

Generate many pseudo-experiments from the best-fit line with the quoted errors, refit each one (using the fast analytic solution) and histogram $\chi^2_{\min}$. It should follow the $\chi^2$ pdf, and the fraction above the observed value should equal the p-value.

In [ ]:
rng = np.random.default_rng(1)
n_toys = 20000
# toy datasets: best-fit line + Gaussian noise with the quoted sigmas, shape (n_toys, 10)
y_toys = ...

M = C_an @ A.T @ W                        # (2,10): maps y -> theta_hat
th_toys = y_toys @ M.T                    # (n_toys, 2)
chi2_toys = ...                           # chi2_min of each toy

plt.hist(chi2_toys, bins=60, range=(0, 20), density=True, alpha=0.5, label="toys")
plt.plot(x, pdf, "b-", label=r"$\chi^2$ pdf")
plt.axvline(chi2_min, color="k", ls="--")
plt.xlabel(r"$\chi^2_{min}$"); plt.legend(); plt.show()
print("toy p-value:", np.mean(chi2_toys >= chi2_min), "  analytic:", pvalue)

✏️ **Exercise 6.** Generate the toys and compute their χ². The two p-values should agree within ~0.01.

## 6. Covariance and correlation

The covariance matrix of the fit parameters has the variances on the diagonal and the covariance off-diagonal.

In [ ]:
cov = np.array(m.covariance)
corr = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1])
print("covariance matrix:\n", cov)
print(f"correlation coefficient rho(theta0, theta1) = {corr:.3f}")

**❓ Question 4.** Why are $\theta_0$ and $\theta_1$ so strongly (anti-)correlated? *Hint:* all $x_i>0$, so the line pivots about a point far from $x=0$. 

✏️ **Exercise 7.** Refit using the shifted variable $x' = x-\bar x_w$, where $\bar x_w$ is the *weighted* mean of $x$. What happens to the correlation? Why is that useful?

In [ ]:
xbar = ...   # weighted mean of x, weights 1/sigma^2

def chi2_shift(a, b):
    ...      # same chi2 but with the model a + b*(x - xbar)

m2 = Minuit(chi2_shift, a=1.0, b=0.5)
...          # set errordef and minimise
c2 = np.array(m2.covariance)
print(f"x_bar = {xbar:.3f}")
print(f"a = {m2.values['a']:.4f} +/- {m2.errors['a']:.4f}")
print(f"b = {m2.values['b']:.4f} +/- {m2.errors['b']:.4f}")
print(f"correlation = {c2[0,1]/np.sqrt(c2[0,0]*c2[1,1]):.3e}")

*(Your observation: what changed for $b$? What changed for $a$?)*

## 7. Confidence regions in the $(\theta_0,\theta_1)$ plane

Two ways to draw them:

1. **Ellipse from the covariance matrix** (parabolic approximation): the contour $\Delta\chi^2 = n^2$ is an ellipse with semi-axes $n\sqrt{\lambda_{1,2}}$ along the eigenvectors of the covariance matrix.
2. **Minos contour** (`mncontour`): follows the *true* $\chi^2_{\min}+\Delta\chi^2$ contour numerically.

**Important:** $\Delta\chi^2=1$ (the "1σ ellipse", $n=1$) is the region whose *projection* onto each axis gives $\pm1\sigma$ on that parameter. It contains only **39 %** of the probability in 2D. A *joint* 68.3 % region needs $\Delta\chi^2=2.30$ (2 dof).

In [ ]:
def get_error_ellipse(cov, center, nstd, **kwargs):
    """Ellipse patch for the contour Delta chi2 = nstd^2.
    cov: 2x2 covariance matrix; center: (mu1, mu2); nstd: scale factor n."""
    eigvals, eigvecs = np.linalg.eigh(cov)            # ascending eigenvalues
    vx, vy = eigvecs[:, 1]                            # eigenvector of the larger eigenvalue
    angle = np.degrees(np.arctan2(vy, vx))
    height, width = 2 * nstd * np.sqrt(eigvals)       # width along the major axis
    return Ellipse(xy=center, width=width, height=height, angle=angle, **kwargs)

In [ ]:
print("Probability inside the n-sigma ellipse (2 parameters):")
for n in (1, 2, 3):
    print(f"  n={n}:  {...:.3f}")     # hint: stats.chi2.cdf(n**2, df=2)

# n such that the ellipse contains a JOINT 68.27 % probability
n_68 = ...                           # hint: sqrt of the chi2 quantile, stats.chi2.ppf(p, df=2)
print(f"n for joint 68.27% = {n_68:.3f}")

fig, ax = plt.subplots(figsize=(6.5, 5))
for n, col in ((3, "green"), (2, "blue"), (1, "red")):
    ax.add_artist(get_error_ellipse(cov, (theta0, theta1), n, fc=col, alpha=0.3, label=f"ellipse n={n}"))
ax.add_artist(get_error_ellipse(cov, (theta0, theta1), n_68, fill=False, ec="k", lw=2, label="ellipse, joint 68.3%"))
for cl in (1, 2, 3):
    pts = np.array(m.mncontour("theta0", "theta1", cl=cl, size=60))
    ax.plot(pts[:, 0], pts[:, 1], "k--", lw=1, label="mncontour" if cl == 1 else None)
ax.plot(theta0, theta1, "k+", ms=12)
ax.set(xlim=(-0.5, 1.6), ylim=(0.3, 1.0), xlabel=r"$\theta_0$", ylabel=r"$\theta_1$")
ax.legend(loc="upper right", fontsize=9); plt.tight_layout(); plt.show()

✏️ **Exercise 8.** Fill in the two missing quantities.

**❓ Question 5.** Which ellipse does the dashed `mncontour(cl=1)` line match best, and why? Is the parabolic (ellipse) approximation good for this problem? *(your answer)*

*Optional:* `m.draw_contour("theta0", "theta1")` and `m.draw_mncontour(...)` give quick plots of the same thing.

## 8. Uncertainty band on the fitted line (error propagation)

The fitted line $\hat y(x)=\hat\theta_0+\hat\theta_1 x$ is a function of correlated parameters, so

$$\sigma^2_{\hat y}(x)=\sigma_0^2+x^2\sigma_1^2+2x\,\mathrm{cov}(\theta_0,\theta_1).$$

The last term is the contribution of the correlation.

In [ ]:
var_full  = ...   # with the covariance term
var_naive = ...   # same, but dropping the covariance term
sigma_full, sigma_naive = np.sqrt(var_full), np.sqrt(var_naive)

fig, ax = plt.subplots()
ax.errorbar(xd, yd, yerr=yd_err, fmt="bo")
ax.plot(xf, yf, "r--")
ax.fill_between(xf, yf - sigma_full, yf + sigma_full, color="red", alpha=0.25, label="with correlation")
ax.plot(xf, yf + sigma_naive, "k:", label="correlation ignored")
ax.plot(xf, yf - sigma_naive, "k:")
ax.set(xlabel="x", ylabel="y", xlim=(0, 6), ylim=(0, 6)); ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

✏️ **Exercise 9.** Write the two variances.

**❓ Question 6.** Where is the band narrowest, and does that match $\bar x_w$ from Exercise 7? At $x=0$, by how much does ignoring the correlation change the error on $\hat y$?

### 8b. Verify by Monte Carlo

Draw parameter pairs from a 2D Gaussian with the fitted mean and covariance, evaluate the line for each pair, and take the standard deviation at each $x$.

In [ ]:
rng = np.random.default_rng(2)
th_samples = ...   # 20000 draws of (theta0, theta1): rng.multivariate_normal(mean, cov, size)
y_samples = ...    # line evaluated on xf for every draw, shape (20000, 300)
sigma_mc = ...     # std over the draws, at each x

plt.plot(xf, sigma_full, label="analytic")
plt.plot(xf, sigma_mc, "--", label="Monte Carlo")
plt.xlabel("x"); plt.ylabel(r"$\sigma_{\hat y}(x)$"); plt.legend(); plt.show()
print("max relative difference:", np.max(np.abs(sigma_mc / sigma_full - 1)))

✏️ **Exercise 10.** Propagate the uncertainties by Monte Carlo. The relative difference should be about 1 % or less.

## 9. Take-home

1. Replace the data by `yd += rng.normal(0, yd_err)` (a fresh random realisation): is χ²/ndf now ≈ 1? Repeat several times.
2. Multiply all `yd_err` by 0.5 and refit. What happens to the parameter errors and to the p-value?
3. Add a quadratic term $\theta_2 x^2$. Does χ² improve significantly? (Compare $\Delta\chi^2$ with the one extra parameter.)